# Model Comparison and Final Selection - Combined

This notebook pulls together the results logged by all four models (Logistic Regression - Member 1, Random Forest - Member 2, XGBoost - Member 3, SVM - Member 4) and compares them on a consistent basis.

**Validation rule used throughout:** the chronological 80/20 hold-out (train on the earlier ~80% of rows, test on the most recent ~20%) is the PRIMARY evaluation, because it matches the dataset proposal and reflects real deployment (predicting future campaigns from past data). Any stratified/random-split result is reported as SECONDARY only, and is not directly comparable to chronological numbers because the two splits have very different test-set class balances.

## 1. Load the shared experiment log

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src.config import EXPERIMENT_LOG, ROOT

FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

log = pd.read_csv(EXPERIMENT_LOG)
pd.set_option("display.max_colwidth", 60)
log

## 2. Split the log into comparable vs. not-directly-comparable rows

Some rows have blank test scores (search-only runs that were never evaluated on the held-out test set) and one row uses a stratified split instead of chronological. These are excluded from the primary comparison and shown separately.

In [ ]:
has_test_score = log["test_pr_auc"].notna()
is_chronological = ~log["notes"].str.contains("stratified split", case=False, na=False)

primary = log[has_test_score & is_chronological].copy()
search_only = log[~has_test_score].copy()
secondary = log[has_test_score & ~is_chronological].copy()

print("PRIMARY comparison rows (chronological, evaluated on test):", len(primary))
print("SEARCH-ONLY rows (no test score, excluded):", len(search_only))
print("SECONDARY rows (stratified split, not directly comparable):", len(secondary))

## 3. Primary comparison table (chronological hold-out, sorted by Test PR-AUC)

In [ ]:
cols = ["model", "cv_pr_auc", "test_pr_auc", "test_roc_auc", "recall", "precision", "f1"]
primary_sorted = primary[cols].sort_values("test_pr_auc", ascending=False).reset_index(drop=True)
primary_sorted

## 4. Chart: Test PR-AUC by model (chronological, primary)

In [ ]:
plt.figure(figsize=(8, 4.5))
colors = ["#2E7D32" if i == 0 else "#4C72B0" for i in range(len(primary_sorted))]
sns.barplot(data=primary_sorted, x="test_pr_auc", y="model", palette=colors)
plt.axvline(primary_sorted["test_pr_auc"].max(), color="grey", ls=":", lw=1)
plt.xlabel("Test PR-AUC (chronological hold-out)")
plt.ylabel("")
plt.title("Model comparison - chronological test PR-AUC (primary metric)")
plt.tight_layout()
plt.savefig(FIG_DIR / "comparison_test_pr_auc.png", dpi=150, bbox_inches="tight")
plt.show()

### Insight -> decision

- Finding: [fill in once SVM is added - currently Logistic Regression leads at 0.5052 test PR-AUC, ahead of XGBoost tuned (0.4603) and Random Forest tuned (0.4459).]
- Decision: [provisional / final model choice]
- Reason: [why - e.g. best chronological score, most interpretable, fastest to serve]

## 5. Secondary comparison: stratified-split rows (not directly comparable to the table above)

In [ ]:
if len(secondary):
    secondary[["model", "cv_pr_auc", "test_pr_auc", "test_roc_auc", "recall", "precision", "f1", "notes"]]
else:
    print("No stratified-split rows currently logged.")

**Why this is shown separately:** a stratified random split mixes early and late campaign rows into both train and test, so its test set does not have the same class balance as the chronological test set (the real deployment scenario). Scores from the two split types should never be compared directly against each other - only within the same split type.

## 6. Search-only rows (hyperparameter search results, no held-out test evaluation)

In [ ]:
if len(search_only):
    search_only[["model", "cv_pr_auc", "notes"]]
else:
    print("No search-only rows currently logged.")

## 7. Validation strategy used by each model

| Model | Split | Imbalance handling | Tuning method |
|---|---|---|---|
| Logistic Regression | Chronological 80/20 (+ stratified 5-fold CV within training) | class_weight='balanced' | GridSearchCV |
| XGBoost | Chronological 80/20 (primary) + stratified split (secondary) | scale_pos_weight (14.69 / 7.88 depending on split) | RandomizedSearchCV vs Optuna |
| Random Forest | Chronological 80/20 (shared chronological_split from src/lr_model.py) | balanced_subsample (selected after comparing no-handling, class weighting, and SMOTE) | RandomizedSearchCV with stratified 5-fold CV |
| SVM | [pending] | [pending] | [pending] |

All three implemented models use the SAME chronological_split() function and the SAME fixed random_state (42), so the train/test boundary is identical across models - this makes the chronological comparison in Section 3 fair. Each model uses a different imbalance-handling technique; this is expected and appropriate, since each technique is the natural/standard approach for that model family (not an inconsistency).

## 8. Current standing and final model selection

**Status as of this notebook:** 3 of 4 required models are complete and logged (Logistic Regression, XGBoost, Random Forest). SVM (Member 4) is still pending.

**Provisional leader:** Logistic Regression, with the highest chronological test PR-AUC (0.5052), ahead of tuned XGBoost (0.4603) and tuned Random Forest (0.4459). A possible explanation: this dataset's test period has a strong temporal shift from the training period, and a well-regularised linear model can generalise more robustly across that shift than a more flexible model that was tuned primarily on the training-period distribution.

**Final selection:** [TO BE COMPLETED AS A GROUP once SVM results are available - fill in: chosen model, the criteria used (PR-AUC, stability, interpretability, speed), and the reasoning.]

## 9. Summary table for the report and presentation

In [ ]:
summary = primary_sorted.copy()
summary.insert(0, "rank", range(1, len(summary) + 1))
summary